# Target Guided Ordinal Encoding

Target guided ordinal encoding gives order to a categorical column using the target variable.

Here, we order `City` based on the average `Price`.

Lower average price gets a smaller encoded value. Higher average price gets a larger encoded value.

**Important:** This method uses the target column, so use it carefully to avoid target leakage. In real projects, calculate this only on training data.


## 1. Import Libraries

We use pandas for data handling and `OrdinalEncoder` for encoding.


In [6]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

## 2. Create Sample Data

`City` is the categorical input column.

`Price` is the target column used to decide the order.


In [7]:
# 1. Create a dummy dataset
data = {
    "City": ["London", "Paris", "London", "Berlin", "Paris", "Berlin"],
    "Price": [300, 150, 320, 100, 160, 90],  # Target variable
}
df = pd.DataFrame(data)

## 3. Find Target Mean by Category

We calculate the average `Price` for each city.

Then we sort cities from lowest average price to highest average price.


In [8]:
# find the mean of each city
target_means = df.groupby("City")["Price"].mean()
sorted_target_means = target_means.sort_values(ascending=True)
sorted_list = sorted_target_means.index.to_list()
print(sorted_target_means)
print(sorted_list)

City
Berlin     95.0
Paris     155.0
London    310.0
Name: Price, dtype: float64
['Berlin', 'Paris', 'London']


### Observation

The order is based on target mean:

`Berlin < Paris < London`

So Berlin gets the lowest encoded value and London gets the highest encoded value.


## 4. Apply Ordinal Encoding

The sorted city list is passed to `OrdinalEncoder` as the category order.

So the encoding becomes:

- lowest mean target value = smallest code
- highest mean target value = largest code


In [ ]:
ordinal_encoder = OrdinalEncoder(
    categories=[sorted_list], handle_unknown="use_encoded_value", unknown_value=-1
)
encoded_data = ordinal_encoder.fit_transform(df[["City"]])
encoded_data

array([[2.],
       [1.],
       [2.],
       [0.],
       [1.],
       [0.]])

## 5. Create Encoded DataFrame

We convert the encoded array into a DataFrame with a clear column name.


In [10]:
encoded_df = pd.DataFrame(encoded_data, columns=["City_encoded"])
encoded_df

,City_encoded
0,2.0
1,1.0
2,2.0
3,0.0
4,1.0
5,0.0


## 6. Combine With Original Data

Finally, we join the encoded city column with the original dataset.


In [11]:
# concat
city_encoded_df = pd.concat([df, encoded_df], axis=1)
city_encoded_df

,City,Price,City_encoded
0,London,300,2.0
1,Paris,150,1.0
2,London,320,2.0
3,Berlin,100,0.0
4,Paris,160,1.0
5,Berlin,90,0.0


In [12]:
# for new test city
df = pd.DataFrame({"City": ["New York", "Paris", "Berlin"]})

encoded_data = ordinal_encoder.transform(df[["City"]])
encoded_data

array([[-1.],
       [ 1.],
       [ 0.]])

## Conclusion

Target guided ordinal encoding is useful when a category has a relationship with the target variable.

But it should be used carefully because it uses target information.

In real machine learning workflows, calculate this mapping only from the training data, then apply it to validation or test data.
